In [ ]:
suppressPackageStartupMessages({
    library(monocle3)
    library(MASS)
    library(reshape2)
    library(dplyr)
    library(ggplot2)
    library(grid)
    library(gridExtra)
    library(Matrix)
    library(IRdisplay)
    library(tibble)
    library(ggridges)
    library(pheatmap)
    library(stringr)
    library(scales)
    library(DelayedArray)
    library(hooke)
    library(splines)
    library(htmlwidgets)
    library(piano)
    library(lme4)
    library(lmerTest)
    library(minpack.lm)
    library(glmnet)
    library(msigdbr)
    library(rmcorr)
    library(ComplexHeatmap)
    library(circlize)
    library(viridis)
    library(glmmTMB)
    library(ggeffects)
})

In [ ]:
pseudobulk_cds = function(cds_sub, sample_group = "Cluster_group", cell_group = "Grouping") {
    cf_ccs = new_cell_count_set(cds_sub,
                            sample_group = sample_group,
                            cell_group = cell_group)
    
    pb_cds = hooke:::pseudobulk_ccs_for_states(cf_ccs)
    
    return(pb_cds)
    
}

In [ ]:
aggregate_gene_score = function(cds, gene_list=c("CCNB1", "CCNB2","CDK1"), name = "gene_list_score"){
    cds_gene_list_norm = cds[rowData(cds)$gene_short_name %in% gene_list,] 
    aggregate_marker_expression = counts(cds_gene_list_norm)
    aggregate_marker_expression = t(t(aggregate_marker_expression) / colData(cds_gene_list_norm)$Size_Factor)
    aggregate_marker_expression = Matrix::colSums(aggregate_marker_expression)
    aggregate_marker_expression = log(aggregate_marker_expression + 1)
    colData(cds)[[name]] = aggregate_marker_expression
    return(cds)
}

normalize <- function(x) {
    return((x- min(x)) /(max(x)-min(x)))
}

detect_genes_fixed = function (cds, min_expr = 0) {
    assertthat::assert_that(methods::is(cds, "cell_data_set"))
    assertthat::assert_that(is.numeric(min_expr))
    mat_bin <- counts(cds) > min_expr
    rowData(cds)$num_cells_expressed <- Matrix::rowSums(mat_bin)
    colData(cds)$num_genes_expressed <- Matrix::colSums(mat_bin)
    cds
}


estimate_cell_cycle = function(cds, g1s_markers, g2m_markers){
    cds_g1s = cds[rowData(cds)$gene_short_name %in% g1s_markers,] 
    aggregate_g1s_expression = counts(cds_g1s)
    aggregate_g1s_expression = t(t(aggregate_g1s_expression) / colData(cds_g1s)$Size_Factor)
    aggregate_g1s_expression = Matrix::colSums(aggregate_g1s_expression)

    cds_g2m = cds[rowData(cds)$gene_short_name %in% g2m_markers,] 
    aggregate_g2m_expression = counts(cds_g2m)
    aggregate_g2m_expression = t(t(aggregate_g2m_expression) / colData(cds_g2m)$Size_Factor)
    aggregate_g2m_expression = Matrix::colSums(aggregate_g2m_expression)
    cds$g1s_score = log(aggregate_g1s_expression+1)
    cds$g2m_score = log(aggregate_g2m_expression+1)
    cds$proliferation_index = log(aggregate_g1s_expression + aggregate_g2m_expression + 1)
    return(cds)
}

In [ ]:
gene_set_BP = msigdbr::msigdbr(species = "mouse", subcategory = "GO:BP")
gene_set_CC = msigdbr::msigdbr(species = "mouse", subcategory = "GO:CC")
gene_set_hallmarks = msigdbr::msigdbr(species = "mouse", category = "H")
gene_set_reactome = msigdbr::msigdbr(species = "mouse", subcategory = "CP:REACTOME")
gene_set_KEGG = msigdbr::msigdbr(species = "mouse", subcategory = "CP:KEGG_LEGACY")

gene_set_BP_df = gene_set_BP %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_CC_df = gene_set_CC %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_hallmarks_df = gene_set_hallmarks %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_reactome_df = gene_set_reactome %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

gene_set_KEGG_df = gene_set_KEGG %>% 
                dplyr::distinct(gs_name, gene_short_name=gene_symbol) %>% 
                as.data.frame()

In [ ]:
cc.genes = readRDS("../data/cc.genes.RDS")
cc.genes$s.genes = stringr::str_to_title(cc.genes$s.genes)
cc.genes$g2m.genes = stringr::str_to_title(cc.genes$g2m.genes)

## Load data and pseudobulk

In [ ]:
cds_VCS = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_scores_by_embryo_30_all_genes_umap2_added_wo_SiRDNA.RDS")


In [ ]:
cds_VCS = estimate_cell_cycle(cds_VCS, g1s_markers = cc.genes$s.genes, g2m_markers = cc.genes$g2m.genes)

In [ ]:
cds_VCS$Grouping = paste(cds_VCS$Experiment, cds_VCS$Embryo, sep="-")
cds_VCS$Cluster_group = paste0(cds_VCS$trajectory_refined, "_", cds_VCS$Cluster_sub)

In [ ]:
cds_VCS_nuc_ratio = cds_VCS[, cds_VCS$Marker=="Nucleolin" & cds_VCS$Phenotype == "Ratio"]

In [ ]:
pb_cds_nuc_frac_all = pseudobulk_cds(cds_VCS_nuc_ratio)

In [ ]:
metadata = as.data.frame(colData(cds_VCS_nuc_ratio))
pb_metadata = as.data.frame(colData(pb_cds_nuc_frac_all))

pb_scores = pb_metadata %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, cell_group, sample, avg_score) %>%
    arrange(cell_group, sample) %>%
    select(Grouping = cell_group, Cluster_group = sample, avg_score)


orig_scores = metadata %>%
    distinct(Grouping, Cluster_group, .keep_all=T) %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, Grouping, Cluster_group, avg_score) %>%
    arrange(Grouping, Cluster_group) %>%
    select(Grouping, Cluster_group, avg_score)


options(repr.plot.width=3, repr.plot.height=3, repr.plot.res=300)
score_comp_df = orig_scores %>%
    left_join(., pb_scores, by = c("Grouping", "Cluster_group")) %>%
    rename("avg_score_orig" = "avg_score.x", "avg_score_pb" = "avg_score.y") 

score_comp_df %>%
    ggplot(., aes(avg_score_orig, avg_score_pb)) + 
        geom_point(size = 0.5) + 
        monocle3:::monocle_theme_opts()

In [ ]:
metadata_subset = metadata %>%
    distinct(Grouping, Cluster_group, .keep_all=T) %>%
    select(Experiment, Embryo, Sex, trajectory_refined, Cluster_sub, 
           cell_group = Grouping, sample = Cluster_group, avg_score) 

metadata_subset = pb_metadata %>%
    select(cell_group, sample) %>%
    left_join(., metadata_subset, by = c("cell_group", "sample"))

pb_cds_nuc_frac_all$Experiment = NULL
pb_cds_nuc_frac_all$Embryo = NULL
pb_cds_nuc_frac_all$Sex = NULL
pb_cds_nuc_frac_all$trajectory_refined = NULL
pb_cds_nuc_frac_all$Cluster_sub = NULL
pb_cds_nuc_frac_all$avg_score = NULL

pb_cds_nuc_frac_all$Experiment = metadata_subset$Experiment
pb_cds_nuc_frac_all$Embryo = metadata_subset$Embryo
pb_cds_nuc_frac_all$Sex = metadata_subset$Sex
pb_cds_nuc_frac_all$trajectory_refined = metadata_subset$trajectory_refined
pb_cds_nuc_frac_all$Cluster_sub = metadata_subset$Cluster_sub
pb_cds_nuc_frac_all$avg_score = metadata_subset$avg_score

pb_metadata = as.data.frame(colData(pb_cds_nuc_frac_all))

pb_scores = pb_metadata %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, cell_group, sample, avg_score) %>%
    arrange(cell_group, sample) %>%
    select(Grouping = cell_group, Cluster_group= sample, avg_score)

orig_scores = metadata %>%
    distinct(Grouping, Cluster_group, .keep_all=T) %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, Grouping, Cluster_group, avg_score) %>%
    arrange(Grouping, Cluster_group) %>%
    select(Grouping, Cluster_group, avg_score)


options(repr.plot.width=3, repr.plot.height=3, repr.plot.res=300)
score_comp_df = orig_scores %>%
    left_join(., pb_scores, by = c("Grouping", "Cluster_group")) %>%
    rename("avg_score_orig" = "avg_score.x", "avg_score_pb" = "avg_score.y") 

score_comp_df %>%
    ggplot(., aes(avg_score_orig, avg_score_pb)) + 
        geom_point(size = 0.5) + 
        monocle3:::monocle_theme_opts()

## Load lineage data with pseudotime (cells are re-clustered and scores are re-calculated)

In [ ]:
cds_VCS_pseudotime = readRDS("../data/cds_VCS_with_cell_type_labels_E15-15.5_1e-4_subclustered_scores_by_embryo_30_all_genes_pseudotime_wo_SiRDNA.RDS")

In [ ]:
cds_VCS_pseudotime

In [ ]:
cds_VCS_pseudotime$trajectory_refined = cds_VCS_pseudotime$trajectory_label
cds_VCS_pseudotime$Cluster_sub = cds_VCS_pseudotime$Cluster_sub_2

In [ ]:
cds_VCS_pseudotime$Grouping = paste(cds_VCS_pseudotime$Experiment, 
                                    cds_VCS_pseudotime$Embryo, sep="-")
cds_VCS_pseudotime$Cluster_group = paste0(cds_VCS_pseudotime$trajectory_refined, "_", 
                                          cds_VCS_pseudotime$Cluster_sub)

In [ ]:
unique(cds_VCS_pseudotime$trajectory_label)

In [ ]:
cds_VCS_pseudotime_nuc_ratio = cds_VCS_pseudotime[, cds_VCS_pseudotime$Marker=="Nucleolin" & 
                                                      cds_VCS_pseudotime$Phenotype == "Ratio"]

In [ ]:
## Extract myeloid and epithelial lineages from the global cds
cds_VCS_nuc_ratio_sub = cds_VCS_nuc_ratio[, 
            cds_VCS_nuc_ratio$trajectory_refined %in% c("White_blood_cells", "Epithelium")]

In [ ]:
cds_VCS_nuc_ratio_sub
cds_VCS_pseudotime_nuc_ratio

In [ ]:
rowdata = as.data.frame(rowData(cds_VCS_pseudotime_nuc_ratio))
coldata = bind_rows(as.data.frame(colData(cds_VCS_pseudotime_nuc_ratio)),
                    as.data.frame(colData(cds_VCS_nuc_ratio_sub)))
gene_mat = cbind(counts(cds_VCS_pseudotime_nuc_ratio), counts(cds_VCS_nuc_ratio_sub))

In [ ]:
## Combine these two cds
cds_VCS_combined = new_cell_data_set(expression_data = gene_mat, 
                                     cell_metadata = coldata, 
                                     gene_metadata = rowdata)
cds_VCS_combined

In [ ]:
cds_VCS_combined = estimate_size_factors(cds_VCS_combined)

In [ ]:
pb_cds_nuc_frac = pseudobulk_cds(cds_VCS_combined)

In [ ]:
metadata = as.data.frame(colData(cds_VCS_combined))
pb_metadata = as.data.frame(colData(pb_cds_nuc_frac))

In [ ]:
head(pb_metadata, 3)

In [ ]:
pb_scores = pb_metadata %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, cell_group, sample, avg_score) %>%
    arrange(cell_group, sample) %>%
    select(Grouping = cell_group, Cluster_group = sample, avg_score)

head(pb_scores, 3)

In [ ]:
orig_scores = metadata %>%
    distinct(Grouping, Cluster_group, .keep_all=T) %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, Grouping, Cluster_group, avg_score) %>%
    arrange(Grouping, Cluster_group) %>%
    select(Grouping, Cluster_group, avg_score)

head(orig_scores, 3)

In [ ]:
options(repr.plot.width=3, repr.plot.height=3, repr.plot.res=300)
score_comp_df = orig_scores %>%
    left_join(., pb_scores, by = c("Grouping", "Cluster_group")) %>%
    rename("avg_score_orig" = "avg_score.x", "avg_score_pb" = "avg_score.y") 

score_comp_df %>%
    ggplot(., aes(avg_score_orig, avg_score_pb)) + 
        geom_point(size = 0.5) + 
        monocle3:::monocle_theme_opts()

In [ ]:
metadata_subset = metadata %>%
    distinct(Grouping, Cluster_group, .keep_all=T) %>%
    select(Experiment, Embryo, Sex, trajectory_refined, Cluster_sub, 
           cell_group = Grouping, sample = Cluster_group, avg_score) 

metadata_subset = pb_metadata %>%
    select(cell_group, sample) %>%
    left_join(., metadata_subset, by = c("cell_group", "sample"))

In [ ]:
pb_cds_nuc_frac$Experiment = NULL
pb_cds_nuc_frac$Embryo = NULL
pb_cds_nuc_frac$Sex = NULL
pb_cds_nuc_frac$trajectory_refined = NULL
pb_cds_nuc_frac$Cluster_sub = NULL
pb_cds_nuc_frac$avg_score = NULL

pb_cds_nuc_frac$Experiment = metadata_subset$Experiment
pb_cds_nuc_frac$Embryo = metadata_subset$Embryo
pb_cds_nuc_frac$Sex = metadata_subset$Sex
pb_cds_nuc_frac$trajectory_refined = metadata_subset$trajectory_refined
pb_cds_nuc_frac$Cluster_sub = metadata_subset$Cluster_sub
pb_cds_nuc_frac$avg_score = metadata_subset$avg_score

In [ ]:
pb_metadata = as.data.frame(colData(pb_cds_nuc_frac))

In [ ]:
pb_scores = pb_metadata %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, cell_group, sample, avg_score) %>%
    arrange(cell_group, sample) %>%
    select(Grouping = cell_group, Cluster_group= sample, avg_score)

head(pb_scores, 3)

In [ ]:
orig_scores = metadata %>%
    distinct(Grouping, Cluster_group, .keep_all=T) %>%
    select(Experiment, Embryo, trajectory_refined, Cluster_sub, Grouping, Cluster_group, avg_score) %>%
    arrange(Grouping, Cluster_group) %>%
    select(Grouping, Cluster_group, avg_score)

head(orig_scores, 3)

In [ ]:
options(repr.plot.width=3, repr.plot.height=3, repr.plot.res=300)
score_comp_df = orig_scores %>%
    left_join(., pb_scores, by = c("Grouping", "Cluster_group")) %>%
    rename("avg_score_orig" = "avg_score.x", "avg_score_pb" = "avg_score.y") 

score_comp_df %>%
    ggplot(., aes(avg_score_orig, avg_score_pb)) + 
        geom_point(size = 0.5) + 
        monocle3:::monocle_theme_opts()

## Compute intra-lineage correlations

In [ ]:
celltype_to_keep = unique(pb_cds_nuc_frac$trajectory_refined)
celltype_to_keep

In [ ]:
run_combined_rmcorr <- function(gene_expr, id, score) {
    df_pearson <- data.frame(
        Embryo = id, 
        Score = score, 
        Expr = gene_expr
    )

    df_spearman <- data.frame(
        Embryo = id, 
        Score_Ranked = rank(score, ties.method = "average"), 
        Expr_Ranked = rank(gene_expr, ties.method = "average")
    )

    spearman_res <- tryCatch({
        rmc_s <- rmcorr(participant = Embryo, measure1 = Score_Ranked, measure2 = Expr_Ranked, dataset = df_spearman)
        c(Spearman = rmc_s$r, Spearman_pvalue = rmc_s$p)
    }, error = function(e) c(Spearman = NA, Spearman_pvalue = NA))

    pearson_res <- tryCatch({
        rmc_p <- rmcorr(participant = Embryo, measure1 = Score, measure2 = Expr, dataset = df_pearson)
        c(Pearson = rmc_p$r, Pearson_pvalue  = rmc_p$p)
    }, error = function(e) c(Pearson = NA, Pearson_pvalue = NA))

    return(c(spearman_res, pearson_res))
}

res_df = NULL
for (traj in celltype_to_keep) {
    pb_cds_nuc_frac_sub = pb_cds_nuc_frac[, pb_cds_nuc_frac$trajectory_refined == traj]
    pb_cds_nuc_frac_sub = detect_genes_fixed(pb_cds_nuc_frac_sub, 0.01)

    rowdata = as.data.frame(rowData(pb_cds_nuc_frac_sub))

    genes = rowdata %>%
                filter(num_cells_expressed > 0.25*ncol(pb_cds_nuc_frac_sub))

    cm = normalized_counts(pb_cds_nuc_frac_sub)
    cm = cm[as.character(genes$gene_ID),]

    subject_id = as.factor(pb_cds_nuc_frac_sub$Embryo) 
    score_vec = pb_cds_nuc_frac_sub$avg_score


    rmcorr_combined_results <- apply(as.matrix(t(cm)), 2, run_combined_rmcorr, id = subject_id, score = score_vec)

    gene_cor_df_tmp <- t(rmcorr_combined_results) %>%
      as.data.frame() %>%
      rownames_to_column("gene_ID") %>%
      left_join(., rowdata %>% dplyr::select(gene_ID, gene_short_name), by = "gene_ID") %>%
      mutate(trajectory_refined = traj,
             Spearman_FDR = p.adjust(Spearman_pvalue, method = "BH"),
             Pearson_FDR = p.adjust(Pearson_pvalue, method = "BH")) 
    
    if (is.null(res_df)) {
        res_df = gene_cor_df_tmp
    } else {
        res_df = rbind(res_df, gene_cor_df_tmp)
    }
    
    print(paste0(traj, ' Number of genes: ', nrow(genes)))
    print(pb_cds_nuc_frac_sub)
    flush.console()
     
}

In [ ]:
## Combine correlations using Stouffer's method

lineages_to_omit <- c("GABAergic", "Glutamatergic") # omit neurons from the CNS

stouffer_df <- res_df %>%
    filter(!trajectory_refined %in% lineages_to_omit) %>%

    mutate(Spearman_pvalue = ifelse(Spearman_pvalue == 0, 1e-300, Spearman_pvalue),
           Z_score = sign(Spearman) * qnorm(Spearman_pvalue / 2, lower.tail = FALSE)) %>%

    group_by(gene_ID, gene_short_name) %>%
    summarize(
        mean_Spearman = mean(Spearman, na.rm = TRUE),
        median_Spearman = median(Spearman, na.rm = TRUE),
        k_trajectories = n(),                            
        sum_Z = sum(Z_score, na.rm = TRUE),            
        n_positive_0.2 = sum(Spearman > 0.2, na.rm = TRUE), 
        n_positive_0.3 = sum(Spearman > 0.3, na.rm = TRUE), 
        n_positive_0.4 = sum(Spearman > 0.4, na.rm = TRUE), 
        n_negative = sum(Spearman < -0.3, na.rm = TRUE),    
        .groups = "drop"
    ) %>%

    mutate(
        Stouffer_Z = sum_Z / sqrt(k_trajectories),       
        Meta_Pvalue = 2 * pnorm(-abs(Stouffer_Z)),        
        Meta_FDR = p.adjust(Meta_Pvalue, method = "BH")  
    ) %>%

    arrange(desc(Stouffer_Z))

## Compute global correlation

In [ ]:
cm = normalized_counts(pb_cds_nuc_frac_all)

subject_id = as.factor(pb_cds_nuc_frac_all$Embryo) 
score_vec = pb_cds_nuc_frac_all$avg_score

run_combined_rmcorr <- function(gene_expr, id, score) {
    df_pearson <- data.frame(Embryo = id, Score = score, Expr = gene_expr)

    df_spearman <- data.frame(
        Embryo = id,
        Score_Ranked = rank(score, ties.method = "average"),
        Expr_Ranked = rank(gene_expr, ties.method = "average")
    )

    spearman_res <- tryCatch({
        rmc_s <- rmcorr(participant = Embryo, measure1 = Score_Ranked, measure2 = Expr_Ranked, dataset = df_spearman)
        c(Spearman = rmc_s$r, Spearman_pvalue = rmc_s$p)
    }, error = function(e) c(Spearman = NA, Spearman_pvalue = NA))

    pearson_res <- tryCatch({
        rmc_p <- rmcorr(participant = Embryo, measure1 = Score, measure2 = Expr, dataset = df_pearson)
        c(Pearson = rmc_p$r, Pearson_pvalue = rmc_p$p)
    }, error = function(e) c(Pearson = NA, Pearson_pvalue = NA))

    return(c(spearman_res, pearson_res))
}

rmcorr_results = apply(as.matrix(t(cm)), 2, run_combined_rmcorr, id = subject_id, score = score_vec)

gene_cor_df_all = t(rmcorr_results) %>%
    as.data.frame() %>%
    rownames_to_column("gene_ID") %>%
    left_join(., rowdata %>% dplyr::select(gene_ID, gene_short_name), by = "gene_ID") %>%
    na.omit()
                            
gene_cor_df_all <- gene_cor_df_all %>%
                      mutate(
                        Spearman_FDR = p.adjust(Spearman_pvalue, method = "BH"),
                        Pearson_FDR = p.adjust(Pearson_pvalue, method = "BH")
                      ) 

In [ ]:
stouffer_Z_5_median_0.7_0.4_n_6 = stouffer_df %>%
    filter(Stouffer_Z > 5 & Meta_FDR < 0.01 & n_negative < 2 & n_positive_0.3 > 5) %>%
    filter(round(median_Spearman, 2) >= 0.4) %>%
    left_join(., gene_cor_df_all) %>%
    filter(Spearman > 0.7 & Pearson > 0.6 & Spearman_FDR < 0.01)

stouffer_Z_5_median_0.7_0.4_n_6

In [ ]:
res_df_save = res_df %>%
    left_join(., gene_cor_df_all %>%
                    select(gene_ID, global_cor = Spearman))

In [ ]:
saveRDS(res_df_save, "../data/res_gene_corr_df_intra_inter_joined.RDS")
saveRDS(stouffer_df, "../data/stouffer_df_nucleolar_fraction.RDS")
saveRDS(stouffer_Z_5_median_0.7_0.4_n_6, "../data/nucleolar_gene_df.RDS")

In [ ]:
saveRDS(pb_cds_nuc_frac_all, "../data/pb_cds_nucleolar_fraction_global.RDS")

In [ ]:
saveRDS(pb_cds_nuc_frac, "../data/pb_cds_nucleolar_fraction_pseudotime.RDS")